1. What’s happening right now, and how fast is it moving?

- Epidemic curve: new cases over time (rising, plateau, declining)
- Analyse the reproduction rate
- Amount of time the total cases takes to double

2. Will the health system hold, or will it collapse?
3. Who needs the vaccine most urgently, and are we reaching them?

===================================================

OPERATION HEATHSHIELD

OWID COVID_19 ANALYSIS FOR NIGERIA MINISTRY OF HEALTH

===================================================

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')   #use a clean white background with gridlines
plt.rcParams['figure.figsize'] = (12, 7) #sets default figure size (width=12, height = 7 inches)
plt.rcParams['font.size'] = 13  #default font size
plt.rcParams['axes.labelsize'] = 13  #size of axix labels (x and y labels)
plt.rcParams['xtick.labelsize'] = 12    #size of numbers/ticks on axes
plt.rcParams['ytick.labelsize'] = 12  #size of numbers/ticks on axes
plt.rcParams['legend.fontsize'] = 10  # size of legend text
plt.rcParams['figure.titlesize'] = 16  # size of main chart title
plt.rcParams['savefig.dpi'] = 300   # resolution ofsaved images (300 DPI = high quality)
plt.rcParams['savefig.bbox'] = 'tight'  # removes extra white spaces when saving figures. 

print("Starting Operation HealthShield - COVID-19 Analysis for Nigeria Ministry of Health") 
print("="*90)
print("\n")

In [ ]:
# Load data
df = pd.read_csv("../../data/compact.csv")

df = df[df["code"] == "NGA"]  # keep only rowas where the "code" column equals "NGA", after this line the data(df) contains  only Nigeria's records

# View summary statistics for numerical columns
numeric_columns = df.columns.tolist() # get all column names and convert them into a  normal python list

# # Removes columns thata re not numeric (this is a manual method)
# numeric_columns.remove("country")
# numeric_columns.remove("date")

#OBetter option is this : the below is another way to remove colums that are not numeric:
numeric_columns = df.select_dtypes(include="number").columns.tolist() # automatically selects only numeric columns. No need to manaulaly remove anyhthing.

# View summary statistics
print(df[numeric_columns].describe()) # selects only numeric columns (df [numeric_columns] .describe () give s statistical summary)

In [ ]:
## Data cleaning and transformation

# Convert the date value from text to a proper datetime object - without this dates a re just strings
df["date"] = pd.to_datetime(df["date"]) #e.g - "2020-01-01"  →  datetime(2020, 1, 1)

# Sort the values in the DataFrame using the date column
df = df.sort_values("date").reset_index(drop=True) # .sort_values("date") - ensure date is in chronological order, .reset_index(drop=True) - resets row numbers after sorting
#drop=True removes old indexinstead of keeping it.

print(f"Data loaded: {len(df):,} rows from {df["date"].min().date()} to {df["date"].max().date()}")
#{len(df) - number of rows in datat frame, 
# :, - adds comma formatting
#df['date']- select the date cplumn , .min() - find the earliest date , .date() - removes time part keeps only date, .max() - find the latest date.
print("\n" + "="*50 + "\n")

# Basic cleaning
# for col in numeric_columns:
#     df[col] = df[col].fillna(0)
#
# cum_fill = [
#     "total_cases", "total_deaths", "people_vaccinated_per_hundred",
#     "people_fully_vaccinated_per_hundred", "total_boosters_per_hundred"
# ]
#
# for col in cum_fill:
#     df[col] = df[col].ffill().fillna(0)

====================================================================

Objective 1: What's happening right now, and how fast is it moving?

====================================================================

In [ ]:
print("\n" + "="*70)
print("OBJECTIVE 1: WHAT'S HAPPENING RIGHT NOW?")
print("="*70)

# Epidemic Curve Analysis - Cases over time ( are cases increasing or decreasing?)
plt.figure()  # createsa new blank chart - so plots dont overlap
sns.lineplot(data=df, x="date", y="new_cases_smoothed", linewidth=2) # x="date" -(horizontal axix-time),  y="new_cases_smoothes  " linewidth= 2 means thicker line - This shows how new COVID cases change over time
plt.title("New Cases Over Time (Rising, Plateau, Declining Phase?)")  # adds chart title
plt.show() #displays the plot

#What the graph shows:
#-Rising - outbreak growin
#-Flat- plateau
#-Falling - outbreak declining


#Below is another line chart - it shows R value over time (is the disease spreading or slowing)
plt.figure()
sns.lineplot(data=df, x="date", y="reproduction_rate", linewidth=2)
plt.axhline(1.0, color="black", linestyle="--", label="R > 1 = growing outbreak") #This draws a horizontal line at R = 1
#R > 1 - each person infects more than 1 - outbreak grows.
#R < 1 - outbreak shrinks.
plt.legend()  # - this displays the label
plt.show() # show the plot

#Together the charts answer the question : “Is the outbreak getting better or worse right now?”
#Result: 
#-Chart 1 - Nigeria experienced multiple waves, peaked around early 2022, then declined sharply and stabilized
#-Chart 2- The outbreak went through cycles of growth and decline, but eventually fell below R = 1 and stayed low
# new_cases_smoothed - means a smoothed (averaged) version of daily new cases to remove noise and fluctuations.

### Final Interpretation of the charts:
The outbreak had multiple waves. The largest surge occured around early 2022.
Since then:
- cases have declined sharply
- reproduction rate is below 1
- Current state:
Outbreak is under control/declining

In [ ]:
# Covid19 "Reproduction Rate" Monitoring

plt.figure()
sns.lineplot(data=df, x="date", y="reproduction_rate", linewidth=2)
plt.axhline(1.0, color="black", linestyle="--", label="R > 1 = growing outbreak")
plt.title("Reproduction Rate Monitoring")
plt.ylabel("Reproduction Rate")
plt.xlabel("Date")
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

rep_rate_data = df.dropna(subset=["reproduction_rate", "date"])

ax.axhline(y=1.0, color="black", linestyle="--", label="Rt = 1 (threshold)", linewidth=1.2)

ax.fill_between(rep_rate_data["date"], 1, rep_rate_data["reproduction_rate"], where=(rep_rate_data["reproduction_rate"] >= 1), color="red", alpha=0.3, label="Rt > 1 (growing)")

ax.fill_between(rep_rate_data["date"], 1, rep_rate_data["reproduction_rate"], where=(rep_rate_data["reproduction_rate"] < 1), color="green", alpha=0.3, label="Rt < 1 (shrinking)")

ax.plot(rep_rate_data["date"], rep_rate_data["reproduction_rate"], linewidth=1.5)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
# ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))

plt.xticks(rotation=45)
ax.set_title("Covid19 Reproduction Rate Monitoring")
ax.set_ylabel("Rt")
ax.set_ylim(0, rep_rate_data["reproduction_rate"].quantile(0.99) + 0.3)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

: 

In [ ]:
beds_per_thousand = df["hospital_beds_per_thousand"].dropna().iloc[0]
beds_per_million = beds_per_thousand * 1000
covid_capacity = beds_per_million * 0.20 # Assuming 20% of beds are allocated to Covid patients

hosp_avail = df["hosp_patients_per_million"].notna().sum()
icu_avail = df["icu_patients_per_million"].notna().sum()

if hosp_avail > 10:
    fig, ax = plt.subplots(figsize=(14, 6))
    hosp_data = df.dropna(subset=["hosp_patients_per_million"])

    ax.plot(df["date"], df["hosp_patients_per_million"])
    # ax.axhline(y=covid_capacity, color="red", linestyle="--", label="Est. Surge Capacity")
    ax.set_title("Hospital Capacity Pressure")
    ax.set_ylabel("Patients per million")
    ax.set_xlabel("Date")

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    # ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))

    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

======================================================
Objective 2: Will the health system hold, or will it collapse?
======================================================

In [ ]:
# Hospital Capacity Pressure Index

WA_BPT = 0.6

beds_per_thousand = df["hospital_beds_per_thousand"].dropna().iloc[0] if not df["hospital_beds_per_thousand"].dropna().empty else WA_BPT
population = df["population"].dropna().iloc[0]

total_beds = beds_per_thousand * (population / 1000)

df["total_beds"] = total_beds
df["hosp_patients_est"] = df["hosp_patients_per_million"] * (population / 1_000_000)
df["bed_occupancy_%"] = ((df["hosp_patients_est"] / total_beds) * 100).fillna(0)

# print(df[["country", "date", "total_beds", "hosp_patients_est", "bed_occupancy_%"]].head(30))

plt.figure()
sns.lineplot(data=df, x="date", y="bed_occupancy_%", linewidth=3)
# plt.axhline(60, color="red", linestyle="--", label="Critical threshold (80%)")
plt.title(f"Hospital Capacity Pressure Index\nNigeria has only ~{int(total_beds):,} beds in total")
plt.ylabel("Est. Bed occupancy %")
plt.savefig("Hospital Capacity Pressure Index.png", dpi=300)
plt.show()

In [ ]:
# # ICU Surge Forecasting
#
# recent_df = df.copy().tail(90)
# recent_df["days"] = (recent_df["date"] - recent_df["date"].min()).dt.days
#
# if recent_df["weekly_icu_admissions_per_million"].max() > 0:
#     slope, intercept = np.polyfit(recent_df["days"], recent_df["weekly_icu_admissions_per_million"], 1)
#     # polyfit() method helps you find the best polynomial curve that fits through your data points
#     # linear trend
#     future_days = np.arange(recent_df["days"].max() + 1, recent_df["days"].max() + 29)
#     forecast = slope * future_days + intercept
#
#     plt.figure(figsize=(14, 6))
#     plt.plot(recent_df["date"], recent_df["weekly_icu_admissions_per_million"], label="Historical", linewidth=3)
#     plt.plot(recent_df["date"].max() + pd.to_timedelta(future_days, "D"), forecast, color="red", label="2-4 Week Forecast", linewidth=3)
#     plt.title("ICU Surge Forecasting (2-4 weeks ahead")
#     plt.ylabel("Weekly ICU addmissions per million")
#     plt.legend()
#     plt.show()
# else:
#     print("Incomplete data")

============================================
Objective 3: WHo needs the vaccine most urgently, and are we reaching them?
============================================

In [ ]:
# Vaccination Velocity Tracking
vvt_df = df.dropna(subset=["new_vaccinations_smoothed"])

plt.figure(figsize=(14, 6))
sns.lineplot(data=vvt_df, x="date", y="new_vaccinations_smoothed", linewidth=3)
plt.title("Vaccination Velocity Tracking\n(Is vaccination rollout accelerating or stalling?)")
plt.xlabel("Date")
plt.ylabel("New vaccinations smoothed (avg doses per day)")
plt.show()

In [ ]:
# Vaccination Coverage Gap Analysis - how many persons never returned for 2nd dose
vcga_df = df.dropna(subset=["people_vaccinated_per_hundred", "people_fully_vaccinated_per_hundred"])

print(vcga_df[["people_vaccinated_per_hundred"]])

plt.figure()
sns.lineplot(data=vcga_df, x="date", y="people_vaccinated_per_hundred", linewidth=3)
sns.lineplot(data=vcga_df, x="date", y="people_fully_vaccinated_per_hundred", linewidth=3)
plt.title("Coverage Gap Analysis")
plt.ylabel("% of population")
plt.show()